# Predicting Student Health Risk

Kaggle Playground Series - Season 6, Episode 7.

This cleaned GitHub edition reproduces the main modelling workflow from the original Kaggle notebook. The competition metric is **balanced accuracy** and the final selected model is a full-feature CatBoost classifier.

**Best public leaderboard score:** `0.94922`.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, recall_score
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from catboost import CatBoostClassifier

pd.set_option("display.max_columns", None)

## 1. Load data

The notebook expects the Kaggle competition dataset to be attached. If you run locally, update the paths below.

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Submission:", sample_submission.shape)

## 2. Target distribution

The target has three classes and is strongly imbalanced, so ordinary accuracy is not sufficient.

In [ ]:
target = "health_condition"
print(train[target].value_counts())
print(train[target].value_counts(normalize=True).mul(100).round(2))

## 3. Missing values

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": train.isnull().sum(),
    "missing_percentage": train.isnull().mean().mul(100).round(2)
})
missing_summary = missing_summary[missing_summary.missing_count > 0].sort_values("missing_percentage", ascending=False)
missing_summary

Train and test showed essentially identical missing-value rates. The largest were `stress_level` (12.00%) and `sleep_duration` (11.01%).

## 4. Feature groups

In [ ]:
X = train.drop(columns=["id", target])
y = train[target].copy()
X_test = test.drop(columns=["id"])

numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical:", numerical_features)
print("Categorical:", categorical_features)

## 5. Stratified holdout split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(X_train.shape, X_valid.shape)

## 6. Majority-class benchmark

In [ ]:
majority_predictions = np.full(len(y_valid), y_train.mode()[0])
print("Accuracy:", accuracy_score(y_valid, majority_predictions))
print("Balanced accuracy:", balanced_accuracy_score(y_valid, majority_predictions))

The majority baseline scores about **0.3333 balanced accuracy**, despite ordinary accuracy near 0.86.

## 7. Scikit-learn preprocessing

In [ ]:
numerical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

## 8. Decision Tree baseline

In [ ]:
tree_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=50, class_weight="balanced", random_state=42
    ))
])
tree_pipeline.fit(X_train, y_train)
tree_pred = tree_pipeline.predict(X_valid)
print("Decision Tree balanced accuracy:", balanced_accuracy_score(y_valid, tree_pred))

Original validation balanced accuracy: **0.9017**.

## 9. HistGradientBoosting

In [ ]:
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
hgb = HistGradientBoostingClassifier(
    learning_rate=0.08, max_iter=200, max_leaf_nodes=15,
    min_samples_leaf=50, l2_regularization=1.0,
    early_stopping=True, validation_fraction=0.10,
    n_iter_no_change=20, random_state=42
)
hgb_pipeline = Pipeline([("preprocessor", preprocessor), ("model", hgb)])
hgb_pipeline.fit(X_train, y_train, model__sample_weight=sample_weights)
hgb_pred = hgb_pipeline.predict(X_valid)
print("HGB balanced accuracy:", balanced_accuracy_score(y_valid, hgb_pred))

Best holdout balanced accuracy for the tuned HGB model: **0.9106**. Its 3-fold CV mean was **0.9092 ± 0.0014**. The first HGB public Kaggle score was **0.90524**.

## 10. CatBoost preparation

CatBoost is used directly on the original numerical and categorical features. Categorical missing values are represented explicitly as `Missing`.

In [ ]:
X_train_cb = X_train.copy()
X_valid_cb = X_valid.copy()
X_test_cb = X_test.copy()

for col in categorical_features:
    X_train_cb[col] = X_train_cb[col].fillna("Missing")
    X_valid_cb[col] = X_valid_cb[col].fillna("Missing")
    X_test_cb[col] = X_test_cb[col].fillna("Missing")

## 11. CatBoost validation model

In [ ]:
cat_model = CatBoostClassifier(
    iterations=800, learning_rate=0.08, depth=7,
    loss_function="MultiClass", eval_metric="MultiClass",
    auto_class_weights="Balanced", random_seed=42,
    verbose=100, allow_writing_files=False
)
cat_model.fit(
    X_train_cb, y_train,
    cat_features=categorical_features,
    eval_set=(X_valid_cb, y_valid),
    early_stopping_rounds=75, verbose=100
)
cat_pred = cat_model.predict(X_valid_cb).reshape(-1)
print("Best iteration:", cat_model.get_best_iteration())
print("Balanced accuracy:", balanced_accuracy_score(y_valid, cat_pred))

The extended CatBoost run stopped at **iteration 568** (569 trees) and achieved **0.9497 validation balanced accuracy**. Class recall was approximately 0.9348 (`at-risk`), 0.9485 (`fit`), and 0.9653 (`unhealthy`).

## 12. Feature importance

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": cat_model.get_feature_importance()
}).sort_values("importance", ascending=False)
feature_importance

The most important features were `stress_level`, `sleep_duration`, `bmi`, and `physical_activity_level`.

## 13. Final full-data CatBoost model

In [ ]:
X_full_cb = X.copy()
for col in categorical_features:
    X_full_cb[col] = X_full_cb[col].fillna("Missing")

final_model = CatBoostClassifier(
    iterations=569, learning_rate=0.08, depth=7,
    loss_function="MultiClass", auto_class_weights="Balanced",
    random_seed=42, verbose=100, allow_writing_files=False
)
final_model.fit(X_full_cb, y, cat_features=categorical_features, verbose=100)
final_test_predictions = final_model.predict(X_test_cb).reshape(-1)

## 14. Submission

In [ ]:
submission = sample_submission.copy()
submission[target] = final_test_predictions

assert submission.shape == sample_submission.shape
assert submission["id"].equals(test["id"])
assert submission.isnull().sum().sum() == 0
assert set(submission[target].unique()).issubset(set(y.unique()))

submission.to_csv("submission_catboost.csv", index=False)
submission.head()

## Final result

The full-feature CatBoost submission achieved a **0.94922 public leaderboard score**. A reduced 11-feature CatBoost experiment scored slightly higher on the single local holdout (`0.9498`) but lower on Kaggle (`0.94822`), so the full-feature model was retained as the final selected submission.